# Case Study: LLM-Powered Product (RAG Assistant)

Designing an LLM-powered product has new dimensions that don't appear in classic ML design: context management, grounding, hallucination monitoring, and cost/latency at token granularity. This note compares the LLM system design to classic ML design and walks through a customer-support assistant.

## What Interviewers Test
- Chunking, embedding, and retrieval design for RAG
- Prompt assembly and grounding/citation
- Eval strategy for generation (golden sets, LLM-as-judge)
- Token math for cost and latency budgeting
- Monitoring hallucination rate in production
- Where LLM design differs from classic ML system design

## Classic ML vs LLM System Design Differences

| Dimension | Classic ML | LLM-Powered System |
|---|---|---|
| **Model** | Task-specific, trained in-house | General foundation model, few-shot |
| **Feature engineering** | Tabular features | Prompt engineering, retrieval |
| **Offline eval** | AUC, NDCG on labeled dataset | Human eval, LLM-as-judge, task completion |
| **Online metrics** | CTR, watch-time | CSAT, resolution rate, escalation rate |
| **Primary failure mode** | Distribution shift | Hallucination, context overflow |
| **Cost unit** | Inference FLOPs | Tokens (input + output) |
| **Latency driver** | Model size | Token count × generation speed |


In [ ]:
# Token math calculator
def token_cost_latency(
    n_docs_retrieved=5,
    avg_chunk_tokens=200,
    query_tokens=50,
    system_prompt_tokens=500,
    output_tokens=300,
    model_cost_per_1k_input=0.003,   # $/1k tokens (e.g., GPT-4o)
    model_cost_per_1k_output=0.015,
    tokens_per_second=50,            # typical generation speed
):
    context_tokens = system_prompt_tokens + query_tokens + n_docs_retrieved * avg_chunk_tokens
    total_input    = context_tokens
    total_output   = output_tokens
    
    cost = (total_input * model_cost_per_1k_input / 1000 +
            total_output * model_cost_per_1k_output / 1000)
    
    latency_generation = output_tokens / tokens_per_second  # seconds
    latency_total      = 0.5 + latency_generation            # +500ms for retrieval etc.
    
    print(f"Context tokens: {context_tokens}")
    print(f"Cost per request: ${cost:.4f}")
    print(f"Estimated latency: {latency_total:.1f}s")
    print(f"Requests/day at $100 budget: {100/cost:.0f}")
    return cost, latency_total

print("=== Default (5 docs retrieved) ===")
cost_5, lat_5 = token_cost_latency()
print()
print("=== 10 docs retrieved ===")
cost_10, lat_10 = token_cost_latency(n_docs_retrieved=10)
print()
print("=== With caching (1 doc, prompt cached) ===")
cost_cached, lat_cached = token_cost_latency(
    n_docs_retrieved=1, system_prompt_tokens=0,  # cached
    model_cost_per_1k_input=0.0003  # 90% cache discount
)


## RAG Pipeline Design

```
User query
  ↓ [Query expansion / rewrite (optional)]
  ↓ [Retrieval: BM25 + embedding → hybrid]
  ↓ [Re-ranking: cross-encoder on top-20 → top-5]
  ↓ [Context assembly: chunks + citations]
  ↓ [LLM generation with grounding prompt]
  ↓ [Post-processing: extract answer, add citations]
Response + sources
```

**Chunking strategies:**

| Strategy | Description | Best for |
|---|---|---|
| **Fixed size** | Split every N tokens with overlap | Simple docs, uniform structure |
| **Sentence-level** | Split on sentence boundaries | Narrative text, FAQs |
| **Semantic** | Split on embedding similarity drops | Technical docs, multi-topic |
| **Document structure** | Split on headers/sections | Markdown, HTML, PDFs |


## Eval Strategy

**Golden set:** ~200 curated (question, ideal_answer, relevant_docs) triples. Run against new versions to detect regressions.

**LLM-as-judge (pairwise):** Ask a judge LLM: "Which answer is better, A or B?" Swap order to correct for position bias. Aggregate wins.

**Online signals:**
- Thumbs up/down per response
- Follow-up question rate (low = resolved)
- Escalation to human agent rate
- Session abandonment rate

**Hallucination monitoring:** Sample 1% of responses, check if factual claims are grounded in retrieved docs (LLM judge or regex-based citation check).


## Guardrails & Fallback

```
User input → [Input guardrails: off-topic, PII, jailbreak detection]
             ↓ (pass)
           [RAG + LLM generation]
             ↓
[Output guardrails: hallucination check, policy compliance, PII redaction]
             ↓ (fail: low confidence or policy violation)
[Fallback: route to human agent or return "I don't know" safely]
```

**Caching:** Cache responses to common questions (fuzzy match on embedding). Can serve 30–50% of traffic from cache, dramatically reducing cost and latency.


## Common Interview Questions

**Q: How is designing an LLM product different from classic ML system design?**
The model is fixed (foundation model, not trained by you), so model selection is replaced by prompt engineering and retrieval design. Eval changes from AUC/NDCG to human or LLM-based evaluation. The primary failure mode is hallucination rather than distribution shift. Cost is measured in tokens, not FLOPs. Latency is dominated by token generation, not model forward pass.

**Q: What is RAG and why use it instead of fine-tuning?**
Retrieval-Augmented Generation fetches relevant documents at query time and includes them in the LLM's context. Fine-tuning bakes knowledge into the model weights. RAG is preferred when: knowledge changes frequently, answers require source attribution, domain knowledge is large relative to fine-tuning budget, or knowledge updates are needed without retraining.

**Q: How do you evaluate an LLM system's answer quality?**
Multiple signals: (1) automated golden-set comparison (exact match for factual queries), (2) LLM-as-judge for longer answers (pairwise ranking with position-bias correction), (3) online signals (thumbs, escalation rate), (4) regression tests in CI on a fixed eval suite. No single metric is sufficient — use a combination.

**Q: How do you monitor hallucination in production?**
Sample ~1% of responses. For each, check: (a) are factual claims grounded in the cited documents? (run a separate LLM check), (b) does the response contain any contradictions with retrieved documents? Alert if hallucination rate exceeds a threshold (e.g., >5%). For customer support, add human audits on escalated tickets.

## Key Takeaways
- LLM system design differs: prompt engineering vs feature engineering, token cost vs FLOP cost, hallucination vs drift
- RAG components: chunking → retrieval (hybrid BM25+semantic) → re-ranking → context assembly → generation
- Token cost math: context_tokens × input_rate + output_tokens × output_rate; caching can cut by 90%
- Eval: golden-set (automated), LLM-as-judge (quality), online signals (CSAT, escalation)
- Hallucination monitoring: sample responses, check grounding, set alert threshold
- Guardrails at input and output; fallback to human for low-confidence or policy-violating responses